# ITDA 3rd 학술제 — 소비기한 추출 대회 공식 베이스라인

본 주피터 노트북은 **'ITDA 3rd 학술제: 소비기한 추출 대회'**의 공식 베이스라인 파이프라인입니다.
상품 이미지에서 **소비기한(연·월·일)**을 정확히 추출하는 모델 및 알고리즘을 개발하고 검증할 수 있도록 최소한의 실행 규격과 로컬 평가 환경을 제공합니다.

---

## 대회 개요 및 문제 정의

| 항목 | 내용 |
|---|---|
| **입력** | 상품 포장지/용기 사진 1장 |
| **출력** | `(year, month, day)` 튜플 (예: `('2026', '08', '24')`) |
| **평가 지표** | Exact Match (연, 월, 일 3개 값이 모두 일치해야 정답) |
| **제출 규격** | `submission.csv` (`image_id`, `year`, `month`, `day`) |

---

## 접근 방식 가이드라인 (Mental Map)

소비기한 추출 문제에는 다양한 패러다임과 알고리즘을 적용할 수 있습니다. 아래 예시들을 참고하여 자신만의 독창적인 솔루션을 구축해 보세요.

- **접근 A (OCR + 규칙/ML)**: OCR로 텍스트와 좌표(BBox)를 구한 뒤, 정규식·키워드거리·문맥분석 모델로 소비기한을 선별
- **접근 B (Object Detection + Crop OCR)**: Object Detection 모델(YOLO, DETR 등)로 소비기한 영역을 먼저 Crop 한 뒤 해당 영역만 OCR 수행
- **접근 C (End-to-End VLM)**: Vision-Language Model(Florence-2, Qwen2-VL 등)을 이용해 이미지에서 직접 연·월·일을 추출
- **접근 D (이미지 전처리 강화)**: 각인·도트 폰트 인식을 위해 전처리(Dewarping, Contrast Enhancement, Binarization) 파이프라인 강화

---

## 노트북 구성

- **PART 1**: 환경 설정 및 라이브러리 로드
- **MODULE 1**: 기본 문자 인식(OCR) 엔진
- **MODULE 2**: 기본 날짜 파싱 및 유효성 검증
- **MODULE 3**: Predictor 인터페이스 및 기본 Predictor
- **MODULE 4**: 전체 추론 및 제출(submission.csv) 생성
- **MODULE 5**: 로컬 평가(Validation) 프레임워크
- **MODULE 6**: 실행 및 시각화 디버깅

---
## PART 1 — 환경 설정 및 라이브러리 로드

In [1]:
# ============================================================
# CELL 1: 필수 라이브러리 설치
# ============================================================
!pip install -q "numpy<2" easyocr pandas matplotlib opencv-python-headless Pillow

# 라이브러리 설치 후 정상 적용을 위해 커널(런타임) 재시작을 권장합니다.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음 (CPU 모드로 동작)"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires pyarrow>=7.0, which is not installed.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.17.1 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.3 which is incompatible.
streamlit 1.37.1 requires pandas<3,>=1.3.0, but you have pandas 3.0.5 which is incompatible.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 12.3.0 which is incompatible.


"GPU ���� (CPU ���� ����)"


'nvidia-smi'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [2]:
# ============================================================
# CELL 2: 라이브러리 임포트 및 전역 설정
# ============================================================
# str | Path 형태의 타입 표기를 구버전 파이썬에서도 사용할 수 있게 합니다.
from __future__ import annotations

import os
import re
import glob
import time
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Tuple, List, Optional, Any

import cv2
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# 경로 설정 (본인 환경에 맞게 수정하세요)
# ------------------------------------------------------------
IMAGE_DIR = r"경로를 넣으세요."
SUBMISSION_PATH = "submission.csv"

# ------------------------------------------------------------
# 상수 규약
# ------------------------------------------------------------
# 날짜를 찾지 못한 경우 이 값을 반환합니다. 제출 파일에도 그대로 기록됩니다.
NONE_RESULT: Tuple[str, str, str] = ("NONE", "NONE", "NONE")

# 소비기한으로 인정할 연도 범위입니다. 이 범위를 벗어난 값은 후보에서 제외됩니다.
YEAR_MIN, YEAR_MAX = 2023, 2030

# OCR 입력 이미지 긴 변의 최대 크기 (초과 시 종횡비 유지 축소)
MAX_SIDE = 1280

# 이미지 파일 확장자 목록 (추론과 정답 템플릿 생성에서 공통으로 사용)
IMAGE_EXTS = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')

USE_GPU = torch.cuda.is_available()
print(f"GPU 사용 가능 여부: {USE_GPU}")
print(f"이미지 경로: {IMAGE_DIR}")
print(f"버전 확인 | numpy {np.__version__} | pandas {pd.__version__} | opencv {cv2.__version__}")

GPU 사용 가능 여부: False
이미지 경로: C:\Users\admin\OneDrive\Desktop\학술제 데이터셋\archive (2)\expiry_region_detection_dataset\expiry_region_detection_dataset\images\train
버전 확인 | numpy 1.26.4 | pandas 3.0.5 | opencv 4.11.0


---
## MODULE 1 — 문자 인식(OCR) 엔진

이미지에서 텍스트와 위치 정보를 가져오는 최소한의 OCR 클래스입니다.

In [3]:
# ============================================================
# CELL 3: 문자 인식 클래스 (EasyOCR 예시)
# ============================================================

@dataclass
class TextBox:
    """
    인식된 텍스트 단위입니다.

    bbox       : 네 꼭짓점 좌표 [[x1,y1], [x2,y2], [x3,y3], [x4,y4]] (원본 이미지 기준)
                 기본 Predictor는 사용하지 않지만, 텍스트 위치나 특정 키워드와의 거리 등을
                 판별 근거로 활용하고 싶은 경우 여기서 가져다 쓸 수 있습니다.
    text       : 인식된 문자열
    confidence : 인식 신뢰도 (0~1)
    """
    bbox: list
    text: str
    confidence: float


class BasicOCREngine:
    """EasyOCR을 사용한 최소 동작 OCR 엔진"""

    def __init__(self, languages: Tuple[str, ...] = ('ko', 'en'), gpu: bool = USE_GPU):
        import easyocr
        print("OCR 엔진 초기화 중...")
        # 최초 실행 시 인식 모델을 내려받으므로 시간이 걸릴 수 있습니다.
        self.reader = easyocr.Reader(list(languages), gpu=gpu)
        print("OCR 엔진 준비 완료")

    def read(self, image_path: str | Path) -> List[TextBox]:
        """
        이미지에서 텍스트와 위치를 읽어 TextBox 목록으로 반환합니다.
        읽기에 실패해도 예외를 밖으로 던지지 않고 빈 목록을 반환하여
        전체 추론이 중단되지 않도록 합니다.
        """
        try:
            img_array = np.fromfile(str(image_path), np.uint8)
            img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
            if img is None:
                print(f"이미지를 열 수 없습니다 [{image_path}]")
                return []

            h, w = img.shape[:2]
            max_dim = max(h, w)

            # 이미지의 긴 변이 MAX_SIDE를 초과하는 경우 종횡비를 유지하며 축소합니다.
            # 이는 특정 영역을 잘라내는(Crop) 것이 아니며, 사진 전체의 해상도만 낮추어 OCR 속도를 높이고 메모리 초과를 방지합니다.
            if max_dim > MAX_SIDE:
                scale = MAX_SIDE / float(max_dim)
                new_w = int(round(w * scale))
                new_h = int(round(h * scale))
                resized_img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
            else:
                scale = 1.0
                resized_img = img

            results = self.reader.readtext(resized_img)

            text_boxes = []
            for bbox, text, conf in results:
                # 축소된 이미지 기준 좌표를 원본 이미지 기준 좌표로 복원합니다.
                if scale != 1.0:
                    orig_bbox = [[float(pt[0]) / scale, float(pt[1]) / scale] for pt in bbox]
                else:
                    orig_bbox = [[float(pt[0]), float(pt[1])] for pt in bbox]
                text_boxes.append(TextBox(bbox=orig_bbox, text=text, confidence=float(conf)))
            return text_boxes
        except Exception as e:
            print(f"OCR 읽기 실패 [{image_path}]: {e}")
            return []

---
## MODULE 2 — 날짜 파싱 및 유효성 검증

문자열에서 날짜 패턴을 찾고 달력 기준 유효성을 검증합니다.

In [4]:
# ============================================================
# CELL 4: 정규식 패턴 및 날짜 검증
# ============================================================

# 각 패턴 앞뒤의 (?<!\d) 와 (?!\d) 는 숫자 경계 조건입니다.
# 이 조건이 없으면 바코드나 품목보고번호 같은 긴 숫자열의 일부가
# 날짜로 잘못 인식됩니다. (예: '20130628332176' 안의 '20130628')
DATE_PATTERNS = [
    re.compile(r'(?<!\d)(\d{4})[.\-/\s](\d{1,2})[.\-/\s](\d{1,2})(?!\d)'),  # YYYY.MM.DD
    re.compile(r'(?<!\d)(\d{2})[.\-/\s](\d{1,2})[.\-/\s](\d{1,2})(?!\d)'),  # YY.MM.DD
    re.compile(r'(?<!\d)(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일'),             # YYYY년 MM월 DD일
    re.compile(r'(?<!\d)(\d{4})(\d{2})(\d{2})(?!\d)'),                      # YYYYMMDD
]


def validate_date(year: str, month: str, day: str) -> Optional[Tuple[str, str, str]]:
    """
    달력상 존재하는 유효한 날짜인지 검증합니다.

    - 두 자리 연도는 20XX로 해석합니다. (예: '26' -> 2026)
    - YEAR_MIN ~ YEAR_MAX 범위를 벗어나면 제외합니다.
    - datetime 생성이 실패하면 달력에 없는 날짜이므로 제외합니다.
      (예: 2월 30일, 13월)

    반환값은 자릿수를 맞춘 문자열 튜플입니다. (예: ('2026', '05', '09'))
    """
    try:
        y, m, d = int(year), int(month), int(day)
        if y < 100:
            y += 2000
        if not (YEAR_MIN <= y <= YEAR_MAX):
            return None
        datetime(y, m, d)
        return (f"{y:04d}", f"{m:02d}", f"{d:02d}")
    except (ValueError, TypeError):
        return None


def parse_first_date(text: str) -> Optional[Tuple[str, str, str]]:
    """
    문자열 내에서 처음 나타나는 유효한 날짜를 추출합니다.

    한 문자열 안에 날짜 형태가 여러 개 있을 수 있으므로 모든 매치를 순회합니다.
    앞쪽 매치가 검증을 통과하지 못해도 뒤쪽 매치를 계속 확인합니다.
    (예: '제조 2013.06.28 소비기한 2026.05.29' 에서 2013년은 연도 범위를
     벗어나므로 건너뛰고 2026.05.29를 찾습니다)

    찾지 못하면 None을 반환합니다.
    """
    for pattern in DATE_PATTERNS:
        for match in pattern.finditer(text):
            validated = validate_date(*match.groups())
            if validated:
                return validated
    return None

---
## MODULE 3 — Predictor 인터페이스 및 기본 Predictor

모든 참가자는 `BasePredictor` 규격을 상속받아 자신만의 알고리즘을 구현할 수 있습니다.

In [5]:
# ============================================================
# CELL 5: BasePredictor 규격 및 기본 구현체
# ============================================================

@dataclass
class PredictionResult:
    """
    추론 결과 반환 형식입니다.

    ymd   : 예측한 (연, 월, 일). 찾지 못하면 NONE_RESULT
    info  : 디버깅용 부가 정보 (자유 형식, 선택 사항)
    error : 예외가 발생한 경우의 메시지
    """
    ymd: Tuple[str, str, str] = NONE_RESULT
    info: dict = field(default_factory=dict)
    error: Optional[str] = None


class BasePredictor(ABC):
    """참가자 알고리즘 상속용 추상 클래스"""
    name: str = "base"

    @abstractmethod
    def predict(self, image_path: str | Path) -> PredictionResult:
        pass

    def __call__(self, image_path: str | Path) -> PredictionResult:
        """
        predict()를 호출하되 예외를 여기서 처리합니다.
        한 장에서 오류가 나도 NONE_RESULT로 대체되어 전체 추론이 멈추지 않습니다.
        predict()가 튜플을 반환해도 PredictionResult로 변환됩니다.
        """
        try:
            res = self.predict(image_path)
            if isinstance(res, PredictionResult):
                return res
            return PredictionResult(ymd=tuple(res))
        except Exception as e:
            return PredictionResult(error=f"{type(e).__name__}: {e}")


class BasicPredictor(BasePredictor):
    """기본 예시: OCR 결과 중 처음 발견된 유효 날짜를 반환하는 최소 베이스라인"""
    name = "basic_ocr_baseline"

    def __init__(self, engine: BasicOCREngine):
        self.engine = engine

    def predict(self, image_path: str | Path) -> PredictionResult:
        boxes = self.engine.read(image_path)
        # 인식된 텍스트를 순서대로 확인하며 가장 먼저 나오는 유효 날짜를 채택합니다.
        # 제조일자가 소비기한보다 먼저 인식되면 그대로 반환되므로,
        # 어떤 후보를 고를지 판단하는 로직이 성능을 좌우합니다.
        for box in boxes:
            date_found = parse_first_date(box.text)
            if date_found:
                return PredictionResult(ymd=date_found, info={"source": box.text})
        return PredictionResult(ymd=NONE_RESULT)

In [6]:
# ============================================================
# CELL 6: [참가자용] 나만의 커스텀 Predictor 작성 템플릿
# ============================================================

class MyCustomPredictor(BasePredictor):
    """
    [참가자 구현 영역]
    VLM, Object Detection, 커스텀 ML 모델, 딥러닝 파이프라인 등
    자신의 알고리즘을 이 클래스 안에 작성하세요.

    지켜야 할 규격은 predict()의 반환 형식 하나뿐입니다.
    PredictionResult 또는 ('YYYY', 'MM', 'DD') 튜플을 반환하면
    이후 파이프라인은 수정 없이 그대로 동작합니다.
    """
    name = "my_custom_algorithm"

    def __init__(self):
        # 모델, 전처리 모듈, 가중치 로드 등
        pass

    def predict(self, image_path: str | Path) -> PredictionResult:
        # 1. 이미지 읽기 및 알고리즘 적용
        # 2. ('YYYY', 'MM', 'DD') 형태로 반환 (찾지 못한 경우 NONE_RESULT)
        return PredictionResult(ymd=NONE_RESULT)


# 작성한 Predictor를 사용하려면 아래처럼 교체하면 됩니다.
# predictor = MyCustomPredictor()

---
## MODULE 4 — 전체 추론 및 제출(submission.csv) 생성

폴더 내 전체 이미지를 추론하고 `submission.csv` 파일을 생성합니다.

In [7]:
# ============================================================
# CELL 7: 전체 데이터셋 추론 파이프라인
# ============================================================

def list_image_paths(image_dir: str | Path, limit: Optional[int] = None) -> List[str]:
    """폴더 내 이미지 경로를 확장자별로 모아 정렬된 목록으로 반환합니다."""
    paths = []
    for ext in IMAGE_EXTS:
        paths.extend(glob.glob(os.path.join(str(image_dir), ext)))
    # 대소문자 확장자가 같은 파일을 가리키는 환경을 고려해 중복을 제거합니다.
    paths = sorted(set(paths))
    if limit:
        paths = paths[:limit]
    return paths


def run_inference(image_dir: str | Path,
                  predictor: BasePredictor,
                  output_path: str = SUBMISSION_PATH,
                  limit: Optional[int] = None) -> pd.DataFrame:
    """
    폴더 내 모든 이미지를 추론하여 submission.csv 생성

    컬럼: image_id, year, month, day
    image_id는 확장자를 제외한 파일명입니다.
    """
    paths = list_image_paths(image_dir, limit)

    if not paths:
        print(f"[경고] {image_dir} 에서 이미지를 찾지 못했습니다.")
        return pd.DataFrame()

    print(f"총 {len(paths)}장 추론 시작 (Predictor: {predictor.name})")
    rows = []
    start_time = time.time()

    for idx, path in enumerate(paths, 1):
        res = predictor(path)
        year, month, day = res.ymd
        rows.append({
            "image_id": Path(path).stem,
            "year": year,
            "month": month,
            "day": day
        })
        if idx % 50 == 0 or idx == len(paths):
            print(f"  [{idx}/{len(paths)}] 진행 중... ({time.time() - start_time:.1f}초)")

    df = pd.DataFrame(rows)
    # utf-8-sig로 저장하면 엑셀에서 한글이 깨지지 않습니다.
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n제출 파일 저장 완료: {output_path}")
    return df

---
## MODULE 5 — 로컬 평가(Validation) 프레임워크

정답 파일(Ground Truth)과 예측 결과를 비교하여 Exact Match 정답률 및 성능 지표를 측정합니다.

In [8]:
# ============================================================
# CELL 8: 정답 템플릿 생성 및 평가 메트릭 계산
# ============================================================

def make_ground_truth_template(image_dir: str | Path,
                               output_path: str = "ground_truth.csv",
                               limit: Optional[int] = 50) -> pd.DataFrame:
    """
    검증용 정답 라벨 작성 템플릿 CSV 생성

    생성된 파일의 year, month, day 열을 직접 채워 넣으세요.
    값을 채운 행만 채점 대상이 되므로 일부만 라벨링해도 됩니다.
    """
    paths = list_image_paths(image_dir, limit)

    df = pd.DataFrame({"image_id": [Path(p).stem for p in paths],
                       "year": "", "month": "", "day": ""})
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"로컬 평가용 정답 템플릿 생성: {output_path} ({len(df)}행)")
    return df


def _normalize_label_frame(df: pd.DataFrame) -> pd.DataFrame:
    """
    비교 전에 표기를 통일합니다.

    - image_id를 문자열로 맞춥니다.
      (엑셀에서 '000001'이 숫자 1로 저장되는 경우를 대비)
    - year/month/day가 숫자로만 이루어져 있으면 자릿수를 채웁니다.
      (정답에 '5'로 적어도 예측값 '05'와 동일하게 취급)
    """
    out = df.copy()
    out["image_id"] = out["image_id"].astype(str).str.strip()
    for col, width in (("year", 4), ("month", 2), ("day", 2)):
        s = out[col].astype(str).str.strip()
        out[col] = s.where(~s.str.fullmatch(r'\d+', na=False), s.str.zfill(width))
    return out


def evaluate_predictions(predictions, ground_truth):
    """
    Exact Match 및 주요 성능 지표 측정

    predictions, ground_truth 는 CSV 경로 또는 DataFrame을 받습니다.
    (run_inference가 반환한 DataFrame을 그대로 넘겨도 됩니다)

    - Exact Match : 연·월·일 세 값이 모두 일치한 비율 (공식 평가 기준)
    - Coverage    : NONE이 아닌 값을 반환한 비율

    두 지표를 함께 보면 개선 방향을 가늠할 수 있습니다.
    Coverage가 낮으면 날짜를 찾아내는 단계를, Coverage는 높은데
    Exact Match가 낮으면 후보를 고르는 단계를 살펴볼 수 있습니다.
    """
    pred = (pd.read_csv(predictions, dtype=str)
            if isinstance(predictions, (str, Path)) else predictions.copy())
    truth = (pd.read_csv(ground_truth, dtype=str)
             if isinstance(ground_truth, (str, Path)) else ground_truth.copy())

    pred = _normalize_label_frame(pred.fillna("NONE"))
    truth = _normalize_label_frame(truth.fillna(""))

    # 라벨을 채우지 않은 행은 채점에서 제외합니다.
    truth = truth[(truth["year"] != "") & (truth["month"] != "") & (truth["day"] != "")]
    if truth.empty:
        print("채점할 라벨 데이터가 없습니다.")
        return {}

    merged = truth.merge(pred, on="image_id", how="left",
                         suffixes=("_true", "_pred")).fillna("NONE")
    if merged.empty:
        print("정답과 예측의 image_id가 일치하지 않습니다. 파일을 확인하세요.")
        return {}

    exact = (merged["year_true"] == merged["year_pred"]) & \
            (merged["month_true"] == merged["month_pred"]) & \
            (merged["day_true"] == merged["day_pred"])

    coverage = (merged["year_pred"] != "NONE").mean()
    exact_match = exact.mean()

    print("=" * 40)
    print(f"로컬 평가 결과 (총 {len(merged)}건 기준)")
    print(f"- Exact Match (정확도) : {exact_match * 100:.2f}%")
    print(f"- Coverage (응답률)   : {coverage * 100:.2f}%")
    print("=" * 40)

    return {"n": len(merged),
            "exact_match": float(exact_match),
            "coverage": float(coverage)}

---
## MODULE 6 — 실행 및 시각화 디버깅

In [9]:
# ============================================================
# CELL 9: 베이스라인 추론 실행 예시
# ============================================================
ocr_engine = BasicOCREngine()
predictor = BasicPredictor(ocr_engine)

# 1. 추론 실행 (테스트용으로 상위 10개만 실행해보려면 limit=10 설정)
submission_df = run_inference(IMAGE_DIR, predictor, SUBMISSION_PATH, limit=10)
if not submission_df.empty:
    display(submission_df.head(10))

# 2. 로컬 평가 (정답 템플릿을 채운 뒤 주석을 해제하세요)
# make_ground_truth_template(IMAGE_DIR, "ground_truth.csv", limit=50)
# evaluate_predictions(submission_df, "ground_truth.csv")

Using CPU. Note: This module is much faster with a GPU.


OCR 엔진 초기화 중...


c:\Users\admin\anaconda3\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


OCR 엔진 준비 완료
총 10장 추론 시작 (Predictor: basic_ocr_baseline)


c:\Users\admin\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  [10/10] 진행 중... (273.4초)

제출 파일 저장 완료: submission.csv


,image_id,year,month,day
0,000001,NONE,NONE,NONE
1,000002,2025,12,11
2,000003,2026,05,12
3,000004,NONE,NONE,NONE
4,000005,NONE,NONE,NONE
5,000006,NONE,NONE,NONE
6,000007,2025,06,26
7,000008,NONE,NONE,NONE
8,000009,NONE,NONE,NONE
9,000010,NONE,NONE,NONE


In [10]:
# ============================================================
# CELL 10: 이미지 시각화 디버깅 함수
# ============================================================
import matplotlib.pyplot as plt


def visualize_sample(image_path: str | Path, predictor: BasePredictor):
    """이미지 추론 결과 확인용 시각화"""
    img_array = np.fromfile(str(image_path), np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    if img is None:
        print(f"이미지를 열 수 없습니다: {image_path}")
        return
    # OpenCV는 BGR 순서로 읽으므로 matplotlib 표시를 위해 RGB로 변환합니다.
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = predictor(image_path)

    plt.figure(figsize=(8, 8))
    plt.imshow(img_rgb)
    plt.title(f"{Path(image_path).name}\nPrediction: {res.ymd}")
    plt.axis('off')
    plt.show()

    # 어떤 텍스트에서 날짜를 찾았는지 확인하면 오답 원인을 파악하기 쉽습니다.
    if res.info:
        print(f"참조한 텍스트: {res.info}")
    if res.error:
        print(f"오류: {res.error}")


# sample_images = list_image_paths(IMAGE_DIR, limit=1)
# if sample_images:
#     visualize_sample(sample_images[0], predictor)

---
## 알고리즘 발전 과제 (참가자 가이드)

베이스라인의 최소 탐지 로직을 넘어 성능을 향상시키기 위해 다음 과제들을 자유롭게 탐색해 보세요.

1. **인식률 향상**: 찌그러짐, 각인, 도트 폰트 등을 더 잘 인식할 수 있는 이미지 전처리 기법이나 성능이 우수한 OCR / VLM 모델 도입
2. **후보 선별**: 상품 포장지 상의 다양한 날짜(제조일, 품목보고번호, 소비기한) 중 진짜 소비기한을 가려내기 위한 문맥, 키워드, 위치 기반 분류 알고리즘 구축
3. **파이프라인 결합**: Object Detection 모델로 날짜 위치를 영역 Crop 한 후 인식하는 방식과 End-to-End VLM 방식의 장단점 비교 및 앙상블